# PathSnap: 03. HMM & Viterbi Sequential Route Reconstruction
### Combining Machine Learning Probabilities with Road Network Topology

This notebook demonstrates how the Hidden Markov Model (HMM) and Viterbi algorithm convert noisy, fragmented isolated predictions into continuous reconstructed routes:
1. **Emission Probabilities**: Blended Gaussian sensor distance and classifier match confidence.
2. **Transition Probabilities**: Topological connectivity and distance differential consistency $\Delta d = |d_{\text{graph}} - d_{\text{Euclid}}|$.
3. **Exact MAP Viterbi Trellis Decoding**.
4. **Greedy Classifier vs. Viterbi Benchmark** across reserved test trips.
5. **Interactive Geospatial Visualization** comparing raw GPS, predicted route, and reference route.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path(".").resolve().parent))

from src.data.loader import TripLoader
from src.features.candidates import CandidateGenerator
from src.features.engineering import FeatureExtractor
from src.models.train import load_model
from src.models.evaluate import compute_route_continuity
from src.map_matching.hmm import MapMatchingHMM
from src.map_matching.viterbi import viterbi_decode
from src.visualization.maps import create_folium_map, save_folium_map

loader = TripLoader("../data")
cg = CandidateGenerator(search_radius_m=120.0, top_k=6)
fe = FeatureExtractor()
hmm = MapMatchingHMM(sigma_z=20.0, beta=35.0)
model = load_model("../artifacts/model.joblib")


## 1. Sequential Route Reconstruction Benchmark
We evaluate on student-test trips, comparing isolated greedy predictions against Viterbi decoding.


In [ ]:
eval_df = pd.read_csv("../results/evaluation_results.csv")
eval_df


In [ ]:
# Plot Route Continuity Comparison
plt.figure(figsize=(9, 4.5))
x = np.arange(len(eval_df))
width = 0.35
plt.bar(x - width/2, eval_df["continuity_greedy"] * 100, width, label="Classifier Greedy", color="#EF4444")
plt.bar(x + width/2, eval_df["continuity_viterbi"] * 100, width, label="Classifier + HMM Viterbi", color="#10B981")
plt.xlabel("Trip Index")
plt.ylabel("Route Continuity (%)")
plt.title("Impact of HMM Viterbi Decoding on Route Continuity")
plt.xticks(x, [f"Trip {tid}" for tid in eval_df["trip_id"]], rotation=15)
plt.ylim(70, 105)
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()


## 2. Interactive Route Reconstruction Map
Generates an interactive Folium map showing the ground truth route vs. the PathSnap Viterbi predicted trajectory.


In [ ]:
sample_trip_id = eval_df.iloc[0]["trip_id"]
trip = loader.load_trip(sample_trip_id, split="student_test")
track = trip["track"][:250]
cands = cg.generate_candidates(track, trip["arcs"], trip["nodes"])
df_feat = fe.extract_features_for_trip(track, trip["arcs"], trip["nodes"], cands)
probs = model.predict_proba(df_feat[fe.FEATURE_NAMES].values)[:, 1]
df_feat["prob"] = probs
probs_by_point = [df_feat[df_feat["point_idx"] == pt_idx]["prob"].tolist() for pt_idx in range(len(track))]

viterbi_arcs, proj_coords = viterbi_decode(track, cands, trip["arcs"], trip["nodes"], hmm=hmm, classifier_probs_by_point=probs_by_point)

# Extract reference route coordinates: [latitude, longitude] for Leaflet
ref_coords = []
if trip["route"] is not None:
    for arc_id in trip["route"]:
        if arc_id < len(trip["arcs"]):
            u, v = trip["arcs"][arc_id, 0], trip["arcs"][arc_id, 1]
            ref_coords.append([float(trip["nodes"][u, 1]), float(trip["nodes"][u, 0])])
            ref_coords.append([float(trip["nodes"][v, 1]), float(trip["nodes"][v, 0])])

fmap = create_folium_map(track=track, predicted_coords=proj_coords, ground_truth_coords=ref_coords, trip_id=sample_trip_id)
save_folium_map(fmap, "../results/plots/hmm_route_reconstruction.html")
print("Interactive reconstruction map successfully saved to results/plots/hmm_route_reconstruction.html")
